In [12]:
!pip install spacy requests
!python -m spacy download fr_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 MB 1.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [13]:
import requests
import re
import spacy

urls_balzac = [
    "https://www.gutenberg.org/cache/epub/74126/pg74126.txt", # Vol 17 : Bette, Pons
    "https://www.gutenberg.org/cache/epub/55860/pg55860.txt", # Treize, Goriot
    "https://www.gutenberg.org/cache/epub/49482/pg49482.txt", # Grandet, Pierrette
    "https://www.gutenberg.org/cache/epub/71773/pg71773.txt"  # Peau de chagrin, etc.
]


def extraire_et_nettoyer(url):
    response = requests.get(url)
    response.encoding = 'utf-8'
    texte_brut = response.text

    try:
        # On coupe tout ce qui est avant le début du roman
        texte = re.split(r'\*\*\* START OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*', texte_brut, flags=re.IGNORECASE)[1]
        # On coupe tout ce qui est après la fin du roman
        texte = re.split(r'\*\*\* END OF THE PROJECT GUTENBERG EBOOK.*?\*\*\*', texte, flags=re.IGNORECASE)[0]
    except IndexError:
        print(f"Attention, balises non trouvées pour {url}. Utilisation du texte brut.")
        texte = texte_brut

    # Nettoyage des retours à la ligne des txt
    texte = re.sub(r'\r\n', ' ', texte)
    texte = re.sub(r'\n', ' ', texte)
    # Suppression des espaces multiples
    texte = re.sub(r'\s+', ' ', texte)

    return texte.strip()

corpus_balzac = {}
print("Début de l'extraction. Opération 'Densité Sémantique' en cours...\n")

for i, url in enumerate(urls_balzac):
    nom_volume = f"Volume_{i+1}"
    print(f"--> Téléchargement et nettoyage du {nom_volume}...")
    corpus_balzac[nom_volume] = extraire_et_nettoyer(url)
    print(f"    Succès : {len(corpus_balzac[nom_volume])} caractères aspirés.")

print("\nPhase d'extraction terminée. Le corpus est prêt.")

Début de l'extraction. Opération 'Densité Sémantique' en cours...

--> Téléchargement et nettoyage du Volume_1...
    Succès : 1499875 caractères aspirés.
--> Téléchargement et nettoyage du Volume_2...
    Succès : 1259241 caractères aspirés.
--> Téléchargement et nettoyage du Volume_3...
    Succès : 1160426 caractères aspirés.
--> Téléchargement et nettoyage du Volume_4...
    Succès : 1152355 caractères aspirés.

Phase d'extraction terminée. Le corpus est prêt.


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import re
import gc
import spacy

print("0. Allumage du moteur neuronal Spacy...")
nlp = spacy.load("fr_core_news_lg")
nlp.max_length = 5000000

print("1. Préparation du terrain pour l'extraction globale...")

def nettoyer_nom(nom):
    nom = re.sub(r'^[-—\'"«\s]+', '', nom)
    return nom.strip()

# Liste des faux-amis à filtrer
faux_amis = ["Mais", "Mon cher", "Cher ange", "Arrière", "Ma Fanchette", "Monsieur",
             "Madame", "Mademoiselle", "Ah", "Oh", "Eh", "Oui", "Non", "Dieu",
             "Paris", "France", "Alors", "Puis", "Quel"]

# Création du Graphe GLOBAL
G_total = nx.Graph()
chunk_size = 500000

print(f"2. Lancement de l'analyse sur {len(corpus_balzac)} volumes. Opération 'Matrice de Balzac'...\n")

for nom_vol, texte in corpus_balzac.items():
    print(f"--- Analyse du {nom_vol} ({len(texte)} caractères) ---")

    # Découpage du texte en blocs
    chunks = [texte[i:i + chunk_size] for i in range(0, len(texte), chunk_size)]

    for n, chunk in enumerate(chunks):
        print(f"  Traitement du bloc {n+1}/{len(chunks)}...")

        # Traitement NLP du bloc
        doc_chunk = nlp(chunk)

        # Analyse des phrases du bloc
        for phrase in doc_chunk.sents:
            persos_phrase = [nettoyer_nom(ent.text) for ent in phrase.ents if ent.label_ == "PER"]
            persos_phrase = [nom for nom in persos_phrase if len(nom) > 2 and nom not in faux_amis]
            persos_phrase = list(set(persos_phrase)) # Supprimer les doublons de la phrase

            # Mise à jour du graphe GLOBAL
            for i in range(len(persos_phrase)):
                for j in range(i + 1, len(persos_phrase)):
                    p1, p2 = persos_phrase[i], persos_phrase[j]

                    if G_total.has_edge(p1, p2):
                        G_total[p1][p2]['weight'] += 1
                    else:
                        G_total.add_edge(p1, p2, weight=1)

        del doc_chunk
        gc.collect()

print("\nTRAVAIL TERMINÉ. BALZAC EST HACKÉ.")
print(f"-> Le réseau global contient {G_total.number_of_nodes()} personnages connectés par {G_total.number_of_edges()} relations.")


print("\n3. Génération de la visualisation globale...")

# On ne garde que ceux qui ont au moins 15 connexions
seuil_connexions = 15
personnages_importants = [noeud for noeud, degre in G_total.degree() if degre >= seuil_connexions]
G_filtre = G_total.subgraph(personnages_importants)

print(f"Affichage des {len(G_filtre.nodes())} personnages principaux (>{seuil_connexions} connexions)...")

plt.figure(figsize=(20, 14))
pos = nx.spring_layout(G_filtre, k=0.4, iterations=40) # k gère l'espacement entre les nœuds
nx.draw_networkx_nodes(G_filtre, pos, node_color='lightcoral', node_size=600, alpha=0.9)
nx.draw_networkx_edges(G_filtre, pos, alpha=0.15, edge_color='gray')
nx.draw_networkx_labels(G_filtre, pos, font_size=10, font_family='sans-serif', font_weight='bold')

plt.title("La Matrice Sociale de La Comédie Humaine (Graphe Global)", fontsize=20, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

0. Allumage du moteur neuronal Spacy...
1. Préparation du terrain pour l'extraction globale...
2. Lancement de l'analyse sur 4 volumes. Opération 'Matrice de Balzac'...

--- Analyse du Volume_1 (1499875 caractères) ---
  Traitement du bloc 1/3...


In [ ]:
print("Calcul de la densité sémantique (Ratio Mots Porteurs de Sens / Mots Totaux)...")

# On prend un échantillon
echantillon = corpus_balzac["Volume_2"][:100000]
doc_echantillon = nlp(echantillon)

# On filtre la ponctuation et les espaces
tokens_valides = [token for token in doc_echantillon if not token.is_punct and not token.is_space]

# On compte les mots "pleins" (Noms, Verbes, Adjectifs, adverbes)
mots_pleins = [token for token in tokens_valides if token.pos_ in ['NOUN', 'VERB', 'ADJ', 'ADV']]

ratio_densite = len(mots_pleins) / len(tokens_valides)

print(f"-> Sur l'échantillon testé, l'algorithme révèle une densité sémantique de : {ratio_densite:.2%}")

In [ ]:
print("Calcul de la centralité des personnages (Qui tient le réseau ?)...")

centralite = nx.betweenness_centrality(G_total)
top_boss = sorted(centralite.items(), key=lambda x: x[1], reverse=True)[:5]

print("\nLe Top 5 des personnages les plus influents du graphe :")
for rank, (nom, score) in enumerate(top_boss):
    print(f"{rank+1}. {nom} (Score d'influence : {score:.4f})")

In [ ]:
def calculer_richesse_lexicale(doc):
    tokens = [token.text.lower() for token in doc if token.is_alpha]
    ttr = len(set(tokens)) / len(tokens)
    return ttr

for nom, texte in corpus_balzac.items():
    # On prend un échantillon de 50 000 mots pour comparer
    doc_sample = nlp(texte[:200000])
    richesse = calculer_richesse_lexicale(doc_sample)
    print(f"Richesse lexicale de {nom} : {richesse:.4f}")

In [ ]:
from collections import Counter

def extraire_mots_haute_densite_v2(doc):
    # On filtre : Noms/Adjectifs, plus de 4 lettres, pas de (le, la, un, etc.)
    mots_cles = [token.lemma_.lower() for token in doc
                 if token.pos_ in ['NOUN', 'ADJ']
                 and len(token.text) > 4
                 and not token.is_stop]

    # On compte les plus fréquents
    frequences = Counter(mots_cles)

    # On retourne le Top 15
    return [mot for mot, freq in frequences.most_common(15)]

doc_sample = nlp(corpus_balzac["Volume_2"][:500000])
top_mots = extraire_mots_haute_densite_v2(doc_sample)
print("\nLes vrais concepts balzaciens :")
print(top_mots)

In [ ]:
longueurs = [len(phrase) for phrase in doc.sents if len(phrase) < 150]

plt.figure(figsize=(10, 6))
plt.hist(longueurs, bins=50, color='maroon', alpha=0.7)
plt.title("Distribution de la longueur des phrases (Structure Balzacienne)")
plt.xlabel("Nombre de mots par phrase")
plt.ylabel("Fréquence")
plt.show()

In [ ]:
import re

print("Initialisation du Compresseur Sémantique v2.0...")

def optimiser_balzac_v2(phrase_texte):
    doc = nlp(phrase_texte)
    mots_gardes = []

    for token in doc:
        if token.pos_ in ['ADJ', 'ADV'] or token.text.lower() in ['profonds', 'longues']:
            continue

        # On garde l'espace original du mot, pour éviter de casser la ponctuation qui était collée (ex: "Goriot,")
        mots_gardes.append(token.text + token.whitespace_)

    # On rassemble la phrase
    texte_final = "".join(mots_gardes)

    # Réparer les élisions cassées (le + voyelle -> l')
    texte_final = re.sub(r'\b([Ll])[ea]\s+([aeiouyAEIOUYhHéèêàâîïôûù])', r"\1'\2", texte_final)
    texte_final = re.sub(r'\b([Dd])e\s+([aeiouyAEIOUYhHéèêàâîïôûù])', r"\1'\2", texte_final)
    texte_final = re.sub(r'\b([Qq])ue\s+([aeiouyAEIOUYhHéèêàâîïôûù])', r"\1'\2", texte_final)

    # Nettoyer les espaces en trop avant la ponctuation
    texte_final = re.sub(r'\s+([.,?!;])', r'\1', texte_final)
    texte_final = re.sub(r'\s+', ' ', texte_final).strip()

    return texte_final

phrase_originale = "Le pauvre père Goriot, vieillard fatigué par les longues années et les profonds chagrins, montait péniblement le sombre escalier de la misérable pension."

print("\n--- AVANT (Version Balzac : 23 mots) ---")
print(phrase_originale)

print("\n--- APRÈS (Version Optimisée par l'IA) ---")
resultat = optimiser_balzac_v2(phrase_originale)
print(resultat)

# Calcul du taux de compression
taux = 100 - (len(resultat.split()) / len(phrase_originale.split()) * 100)
print(f"\n-> Gras littéraire éliminé : {taux:.1f}%")

In [ ]:
import re
from google.colab import files

print("Initialisation du Scalpel Syntaxique...")

def purger_phrase(phrase):
    mots_a_supprimer = set()

    for token in phrase:
        if token.dep_ in ['amod', 'advmod', 'acl', 'appos']:
            for enfant in token.subtree:
                mots_a_supprimer.add(enfant.i)

    mots_gardes = []
    for token in phrase:
        if token.i not in mots_a_supprimer and token.pos_ not in ['PUNCT']:
            mots_gardes.append(token.text + token.whitespace_)

    texte_final = "".join(mots_gardes)

    texte_final = re.sub(r'\b([Ll])[ea]\s+([aeiouyAEIOUYhHéèêàâîïôûù])', r"\1'\2", texte_final)
    texte_final = re.sub(r'\b([Dd])e\s+([aeiouyAEIOUYhHéèêàâîïôûù])', r"\1'\2", texte_final)
    texte_final = re.sub(r'\b([Qq])ue\s+([aeiouyAEIOUYhHéèêàâîïôûù])', r"\1'\2", texte_final)
    texte_final = re.sub(r'\s+', ' ', texte_final).strip()

    if texte_final:
        return texte_final + ". "
    return ""

phrase_test = nlp("Le pauvre père Goriot, vieillard fatigué par les longues années et les profonds chagrins, montait péniblement le sombre escalier de la misérable pension.")
print("\n[Test AVANT] :", phrase_test.text)
print("[Test APRÈS] :", purger_phrase(next(phrase_test.sents)))

# On prend un extrait (les 300 000 premiers caractères du volume 2)
texte_source = corpus_balzac["Volume_2"][:300000]
doc_roman = nlp(texte_source)

print("\nRéécriture du roman en cours... (Destruction de la poésie)")
roman_hacke = ""

for phrase in doc_roman.sents:
    roman_hacke += purger_phrase(phrase)

# Exportation du fichier
nom_fichier = "Balzac_Optimise_v1.txt"
with open(nom_fichier, "w", encoding="utf-8") as f:
    f.write(roman_hacke)

print(f"\n-> Roman réécrit avec succès ! Le fichier {nom_fichier} est prêt.")
# files.download(nom_fichier)

In [ ]:
print("Calcul des métriques de densité sémantique...")

def calculer_densite(texte):
    doc = nlp(texte)
    mots_totaux = [t for t in doc if t.is_alpha]

    info_pure = [t for t in mots_totaux if t.pos_ in ['NOUN', 'PROPN', 'VERB']]

    if len(mots_totaux) == 0: return 0
    return len(info_pure) / len(mots_totaux)

# On compare un échantillon
echantillon_original = texte_source[:50000]
echantillon_hacke = roman_hacke[:50000] # On prend la même taille pour comparer

densite_avant = calculer_densite(echantillon_original)
densite_apres = calculer_densite(echantillon_hacke)

amelioration = ((densite_apres - densite_avant) / densite_avant) * 100
compression = 100 - (len(roman_hacke.split()) / len(texte_source.split()) * 100)

print(f"Densité sémantique (Balzac)   : {densite_avant:.2%}")
print(f"Densité sémantique (Machine)  : {densite_apres:.2%}")
print(f"--------------------------------------------------")
print(f"Taux de compression du texte  : - {compression:.1f} % (Gras éliminé)")
print(f"AMÉLIORATION DE LA SÉMANTIQUE : + {amelioration:.1f} %")